# Lab 15-11 · Digital Control

Four parts, each built on `digital.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.linalg import expm
from digital import zoh, tustin, pi_tustin, sampled_loop, dfreq, dmargins, jury2

## Part 1 · ZOH equivalents two ways

In [2]:
T = 0.1
for num, den in (([1], [1, 1]), ([1], [1, 1, 0])):
    nz, dz = zoh(num, den, T)
    A, B, C, D = signal.tf2ss(num, den)
    n = len(A); M = np.zeros((n + 1, n + 1)); M[:n, :n] = A; M[:n, n:] = B
    E = expm(M * T); Phi, Gam = E[:n, :n], E[:n, n:]
    nz2, dz2 = signal.ss2tf(Phi, Gam, C, D)
    print(f"{num}/{den}: scipy {np.round(nz, 6)} / {np.round(dz, 6)}; matrix exponential {np.round(np.trim_zeros(nz2[0], 'f'), 6)} / {np.round(dz2, 6)}")

[1]/[1, 1]: scipy [0.095163] / [ 1.       -0.904837]; matrix exponential [0.095163] / [ 1.       -0.904837]
[1]/[1, 1, 0]: scipy [0.004837 0.004679] / [ 1.       -1.904837  0.904837]; matrix exponential [0.004837 0.004679] / [ 1.       -1.904837  0.904837]


## Part 2 · The PI loop against sample period

In [3]:
def pi_ctrl(Kp, Ki, T):
    b0, b1 = pi_tustin(Kp, Ki, T)
    def step(e, st):
        u_prev, e_prev = st if st is not None else (0.0, 0.0)
        u = u_prev + b0 * e + b1 * e_prev
        return u, (u, e)
    return step
Ts = [0.1, 0.25, 0.5, 1.0, 1.5, 2.0]          # beyond about 2.5 s the loop gain no longer falls below 1 before Nyquist
os_, pm = [], []
for T in Ts:
    t, y, _ = sampled_loop(([0.5], [10, 1]), pi_ctrl(14, 5, T), T, 80, n_sub=20)
    os_.append(100 * (y.max() - 1))
    Gz = zoh([0.5], [10, 1], T); b0, b1 = pi_tustin(14, 5, T)
    pm.append(dmargins(np.polymul([b0, b1], Gz[0]), np.polymul([1, -1], Gz[1]), T)["pm"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
a1.plot(Ts, os_, "o-"); a1.set_xlabel("T (s)"); a1.set_ylabel("overshoot (%)")
a2.plot(Ts, pm, "o-", label="discrete"); a2.plot(Ts, [72.44 - math.degrees(0.766 * T / 2) for T in Ts], "--", label="72.4 - wc T/2")
a2.set_xlabel("T (s)"); a2.set_ylabel("phase margin (deg)"); a2.legend()
plt.show()
for T, o, p in zip(Ts, os_, pm):
    print(f"T = {T:4}: overshoot {o:5.1f}%, PM {p:5.1f} deg")

T =  0.1: overshoot  13.1%, PM  70.3 deg
T = 0.25: overshoot  14.0%, PM  67.0 deg
T =  0.5: overshoot  15.7%, PM  61.7 deg
T =  1.0: overshoot  20.8%, PM  51.1 deg
T =  1.5: overshoot  29.4%, PM  40.6 deg
T =  2.0: overshoot  72.2%, PM  29.9 deg


## Part 3 · A switch-and-capacitor sampler

*Predict:* more or less overshoot than the continuous 16.3%?

In [4]:
t, y, _ = sampled_loop(([1], [1, 1, 0]), lambda e, st: (1.0 * e, None), 0.5, 30, n_sub=500)
print(f"ideal ZOH: overshoot {100 * (y.max() - 1):.2f}% at {t[np.argmax(y)]:.3f} s")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "sample_hold.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.startswith(("os =", "tpk ="))])
else:
    print("ngspice not installed: skip (CI runs sample_hold.cir through tools/run_spice.py)")

ideal ZOH: overshoot 28.67% at 3.452 s


ngspice: ['os = 2.861320e+01', 'tpk = 3.452754e+00']


## Part 4 · Discretization methods for a lead

In [5]:
a, Tl, wm, T = 4.029085906587338, 0.11259097915610956, 4.424794853217873, 0.2
w = np.linspace(0.3, math.pi / T * 0.98, 2000)
cont = np.degrees(np.angle((1 + 1j * w * a * Tl) / (1 + 1j * w * Tl)))
methods = {
    "forward Euler": signal.cont2discrete(([a * Tl, 1], [Tl, 1]), T, method="euler")[:2],
    "backward Euler": signal.cont2discrete(([a * Tl, 1], [Tl, 1]), T, method="backward_diff")[:2],
    "Tustin": tustin([a * Tl, 1], [Tl, 1], T),
    "prewarped Tustin": tustin([a * Tl, 1], [Tl, 1], T, w_p=wm),
}
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(w, cont, "k", lw=2, label="continuous")
for name, (n, d) in methods.items():
    n = np.trim_zeros(np.atleast_1d(np.squeeze(n)), "f")
    ph = np.degrees(np.angle(dfreq(n, np.asarray(d), w, T)))
    ax.plot(w, ph, label=name)
    print(f"{name:16s}: peak {ph.max():5.1f} deg at {w[np.argmax(ph)]:.2f} rad/s")
ax.set_xlabel("ω (rad/s)"); ax.set_ylabel("phase (deg)"); ax.legend(fontsize=7)
plt.show()

forward Euler   : peak  68.6 deg at 9.82 rad/s
backward Euler  : peak  26.4 deg at 2.84 rad/s
Tustin          : peak  37.0 deg at 4.17 rad/s
prewarped Tustin: peak  37.0 deg at 4.42 rad/s


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.